# Bilan SEO Vilber — analyse T0 / T1 / T2Notebook d'analyse du bilan d'accompagnement Studio Visuweb × Vilber.**Avant de l'exécuter**, lancer la chaîne d'extraction (cf. `README.md`) :```bashpython scripts/extract_search_console.pypython scripts/extract_ga4.pypython scripts/extract_semrush.py --from-csvpython scripts/extract_clarity.pypython scripts/extract_webflow.pypython scripts/consolidate.py```Chaque cellule se dégrade proprement si sa source manque : elle affiche ce quimanque au lieu de planter. On peut donc avancer source par source.⚠️ **Aucun chiffre de ce notebook ne doit partir chez Anaïs sans avoir lu`docs/ECARTS-BRIEF.md`** : plusieurs valeurs de référence du brief necorrespondent pas aux sources Notion.

In [ ]:
import json, sys, warningsfrom pathlib import Pathimport pandas as pdimport matplotlib.pyplot as pltsys.path.insert(0, str(Path.cwd().parent / "scripts"))import vizfrom config import ARTICLES, DATA_CONSOLIDATED, DATA_REFERENCE, FIGURES_DIR, PERIODSviz.apply_style()pd.set_option("display.max_rows", 120)warnings.filterwarnings("ignore", category=FutureWarning)def load(name):    """Charge une table consolidée, ou retourne un DataFrame vide en le disant."""    path = DATA_CONSOLIDATED / f"{name}.csv"    if not path.exists():        print(f"[manquant] {name}.csv — lancer les scripts d'extraction concernés")        return pd.DataFrame()    df = pd.read_csv(path)    print(f"[ok] {name}: {len(df)} lignes")    return dfmanifest = json.loads((DATA_CONSOLIDATED / "manifest.json").read_text()) \    if (DATA_CONSOLIDATED / "manifest.json").exists() else {}print("Sources trouvées :", manifest.get("resume", {}).get("sources_trouvees", "?"),      "| manquantes :", manifest.get("resume", {}).get("sources_manquantes", "?"))

In [ ]:
gsc_totals   = load("gsc_totals")gsc_queries  = load("gsc_queries")gsc_pages    = load("gsc_pages")gsc_daily    = load("gsc_daily")position_mix = load("gsc_position_mix")branded      = load("branded_share")ga4_overview = load("ga4_overview")ga4_channels = load("ga4_channels")ga4_conv     = load("ga4_conversions")clarity      = load("clarity_metrics")webflow      = load("webflow_leads")kpis         = load("kpis_synthese")deltas       = load("deltas")reference = {p.stem: json.loads(p.read_text()) for p in sorted(DATA_REFERENCE.glob("*.json"))}print("\nRéférences Notion chargées :", list(reference))

## 1. Tableau de synthèse T0 / T1 / T2Les volumes sont normalisés **par 30 jours** : T0 fait 31 jours, T1 77, T2 78.Comparer les totaux bruts donnerait une fausse progression d'un facteur ~2,5.

In [ ]:
ORDRE = ["T0", "T1", "T2", "GLOBAL"]if not kpis.empty:    cols = ["kpi"] + [c for c in ORDRE if c in kpis.columns]    synthese = kpis[cols].round(2)    display(synthese)else:    print("Pas encore de KPI consolidé.")if not deltas.empty:    display(deltas.pivot_table(index="kpi", columns="comparaison",                               values="delta_pct").round(1))

In [ ]:
# Rappel des objectifs contractuels (source : base Notion « SEO KPIs & Objectifs »)if "kpis_notion" in reference:    objectifs = pd.DataFrame(reference["kpis_notion"]["kpis"])    display(objectifs[["objectif", "baseline", "cible", "resultat", "statut"]])    print("\n⚠️", reference["kpis_notion"]["avertissement"])

## 2. Courbes mensuelles — clics, impressions, positionTrois graphiques séparés plutôt qu'un double axe : deux échelles superposées surun même cadre font lire des croisements qui n'existent pas.

In [ ]:
if not gsc_daily.empty:    daily = gsc_daily.copy()    daily["date"] = pd.to_datetime(daily["date"])    # GLOBAL recouvre T1 et T2 : on l'exclut pour ne pas compter deux fois.    daily = daily[daily["periode"] != "GLOBAL"]    mensuel = (daily.set_index("date")                    .groupby(pd.Grouper(freq="MS"))                    .agg(clics=("clicks", "sum"),                         impressions=("impressions", "sum"),                         position=("position", "mean"))                    .dropna(how="all"))    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)    for ax, (col, titre, unite) in zip(axes, [        ("clics", "Clics organiques par mois", "clics"),        ("impressions", "Impressions par mois", "impressions"),        ("position", "Position moyenne (plus bas = mieux)", "position"),    ]):        ax.plot(mensuel.index, mensuel[col], color=viz.color_for(unite),                marker="o", label=titre)        ax.set_title(titre)        ax.set_ylabel("")    axes[2].invert_yaxis()   # une position qui s'améliore doit monter    axes[2].set_xlabel("mois")    fig.suptitle("vilber.com — 6 mois d'accompagnement", x=0.02, ha="left",                 fontsize=14, fontweight="semibold")    fig.tight_layout()    viz.save(fig, "01-series-mensuelles")else:    print("gsc_daily manquant — lancer extract_search_console.py")

## 3. Comparaison par période sur les KPIs stratégiques

In [ ]:
if not kpis.empty:    a_tracer = [k for k in kpis["kpi"] if "GLOBAL" not in k]    periodes = [c for c in ["T0", "T1", "T2"] if c in kpis.columns]    n = len(a_tracer)    fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(11, 2.6 * ((n + 1) // 2)))    for ax, kpi in zip(axes.ravel(), a_tracer):        ligne = kpis.loc[kpis["kpi"] == kpi, periodes].iloc[0]        bars = ax.bar(periodes, ligne.values,                      color=[viz.color_for(p) for p in periodes], width=0.55)        viz.label_bars(ax, bars, fmt="{:,.1f}")        ax.set_title(kpi, fontsize=10)        ax.margins(y=0.25)    for ax in axes.ravel()[n:]:        ax.axis("off")    fig.suptitle("KPIs stratégiques par période", x=0.02, ha="left",                 fontsize=14, fontweight="semibold")    fig.tight_layout()    viz.save(fig, "02-kpis-par-periode")

## 4. Part des clics de marque — le KPI stratégique cachéDeux définitions coexistent dans les sources Vilber et ne disent **pas** la mêmechose (cf. `docs/ECARTS-BRIEF.md`, point 3) :- **strict** — `vilber`, `lourmat` seuls. C'est la définition de la base KPIs  Notion (baseline 27,2 %, cible 24 %).- **large** — marque + noms de gammes (`fusion`, `newton`, `e-box`, `quantum`,  `spectra`, `bio-sun`, `bio-link`, `kuant`). C'est la définition de la stratégie  M3 (baseline 70 %, cible 55-60 %).Le rapport doit **nommer la définition qu'il cite**, sinon la même réalité seprésente comme un succès ou comme un échec.

In [ ]:
if not branded.empty:    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)    for ax, definition in zip(axes, ["strict", "large"]):        sub = (branded[branded["definition"] == definition]               .set_index("periode").reindex(ORDRE).dropna(how="all"))        base = ax.bar(sub.index, sub["clics_non_branded"],                      color=viz.color_for("non_branded"), width=0.55,                      label="non-marque")        haut = ax.bar(sub.index, sub["clics_branded"],                      bottom=sub["clics_non_branded"],                      color=viz.color_for("branded"), width=0.55,                      label="marque", edgecolor=viz.SURFACE, linewidth=2)        for x, (nb, b, part) in enumerate(zip(sub["clics_non_branded"],                                              sub["clics_branded"],                                              sub["part_branded_pct"])):            ax.text(x, nb + b, f"{part:.1f}% marque", ha="center", va="bottom",                    fontsize=9, color=viz.INK_SECONDARY)        ax.set_title(f"définition « {definition} »")        ax.margins(y=0.2)    axes[0].set_ylabel("clics")    axes[0].legend(loc="upper left")    fig.suptitle("Clics marque vs non-marque, par période", x=0.02, ha="left",                 fontsize=14, fontweight="semibold")    fig.tight_layout()    viz.save(fig, "03-branded-vs-non-branded")    display(branded.pivot_table(index="periode", columns="definition",                                values="part_branded_pct").reindex(ORDRE))

## 5. Mots-clés gagnants et perdants (T1 → T2)

In [ ]:
def comparer_requetes(df, avant="T1", apres="T2", top=20):    if df.empty or not {avant, apres} <= set(df["periode"].unique()):        print(f"périodes {avant}/{apres} absentes de gsc_queries")        return pd.DataFrame()    piv = df.pivot_table(index="query", columns="periode",                         values=["clicks", "position"], aggfunc="sum")    out = pd.DataFrame({        "clics_avant": piv[("clicks", avant)],        "clics_apres": piv[("clicks", apres)],        "pos_avant": piv[("position", avant)],        "pos_apres": piv[("position", apres)],    }).fillna({"clics_avant": 0, "clics_apres": 0})    out["delta_clics"] = out["clics_apres"] - out["clics_avant"]    out["delta_position"] = out["pos_avant"] - out["pos_apres"]  # positif = progression    return out.sort_values("delta_clics", ascending=False)evolution = comparer_requetes(gsc_queries)if not evolution.empty:    gagnants = evolution.head(20)    perdants = evolution.tail(20).sort_values("delta_clics")    display(gagnants.round(1))    display(perdants.round(1))    fig, axes = plt.subplots(1, 2, figsize=(12, 6))    for ax, data, titre, couleur in [        (axes[0], gagnants.iloc[::-1], "Top 20 gagnants (Δ clics)", viz.DIVERGING["pos"]),        (axes[1], perdants.iloc[::-1], "Top 20 perdants (Δ clics)", viz.DIVERGING["neg"]),    ]:        bars = ax.barh(data.index, data["delta_clics"], color=couleur, height=0.65)        viz.label_bars(ax, bars, fmt="{:+,.0f}")        ax.set_title(titre)        ax.margins(x=0.18)        ax.grid(axis="y", visible=False)    fig.tight_layout()    viz.save(fig, "04-mots-cles-gagnants-perdants")

## 6. Performance par clusterRampe séquentielle à une seule teinte : la couleur code une magnitude, pas uneidentité.

In [ ]:
if not gsc_queries.empty:    par_cluster = (gsc_queries.groupby(["cluster", "periode"])                   .agg(clics=("clicks", "sum"),                        impressions=("impressions", "sum"),                        position=("position", "mean"))                   .reset_index())    display(par_cluster.round(1))    heat = (par_cluster[par_cluster["periode"].isin(["T0", "T1", "T2"])]            .pivot_table(index="cluster", columns="periode", values="clics")            .reindex(columns=[c for c in ORDRE if c != "GLOBAL"]))    fig, ax = plt.subplots(figsize=(6.5, 3.6))    im = ax.imshow(heat.values, cmap=viz.sequential_cmap(), aspect="auto")    ax.set_xticks(range(len(heat.columns)), heat.columns)    ax.set_yticks(range(len(heat.index)), heat.index)    for i in range(heat.shape[0]):        for j in range(heat.shape[1]):            valeur = heat.values[i, j]            if pd.notna(valeur):                seuil = heat.values[~pd.isna(heat.values)].max() * 0.6                ax.text(j, i, f"{valeur:,.0f}", ha="center", va="center",                        fontsize=9,                        color="#ffffff" if valeur > seuil else viz.INK)    ax.grid(visible=False)    ax.set_title("Clics par cluster et par période")    fig.colorbar(im, ax=ax, shrink=0.8, label="clics")    fig.tight_layout()    viz.save(fig, "05-heatmap-clusters")

## 7. Impact des articles publiés (M1 / M2 / M3)Rappel de latence : les articles M3 publiés en juillet-août n'auront leur pleineffet SEO qu'en novembre (T4). Une performance faible en T2 n'est pas un échec,c'est un article qui n'a pas fini de monter.

In [ ]:
articles = pd.DataFrame(ARTICLES)if not gsc_pages.empty:    perf = []    for _, art in articles.iterrows():        match = gsc_pages[gsc_pages["path"].str.contains(art["slug"], case=False, na=False)]        for periode, grp in match.groupby("periode"):            perf.append({                "mois": art["mois"], "titre": art["titre"], "cluster": art["cluster"],                "periode": periode, "clics": grp["clicks"].sum(),                "impressions": grp["impressions"].sum(),                "position": grp["position"].mean(),            })    perf = pd.DataFrame(perf)    if perf.empty:        print("Aucun article retrouvé dans les pages GSC — vérifier les slugs dans config.ARTICLES")    else:        display(perf.round(1))        t2 = perf[perf["periode"] == "T2"].sort_values("impressions", ascending=False)        if not t2.empty:            fig, ax = plt.subplots(figsize=(9, 0.42 * len(t2) + 2))            bars = ax.barh(t2["titre"], t2["impressions"],                           color=[viz.color_for(c) for c in t2["cluster"]], height=0.65)            viz.label_bars(ax, bars, fmt="{:,.0f}")            ax.set_title("Impressions en T2 par article")            ax.grid(axis="y", visible=False)            ax.margins(x=0.18)            handles = [plt.Rectangle((0, 0), 1, 1, color=viz.color_for(c))                       for c in t2["cluster"].unique()]            ax.legend(handles, t2["cluster"].unique(), loc="lower right")            fig.tight_layout()            viz.save(fig, "06-impact-articles")else:    print("gsc_pages manquant")

## 8. Détection d'anomaliesTrois choses à vérifier avant de conclure quoi que ce soit :1. **Le bot Direct de GA4** repéré en T0 et disparu en T1 — est-il stable en T2 ?   S'il revient, la comparaison d'audience est faussée.2. **Les pics et creux quotidiens** de Search Console (seuil : ±2,5 écarts-types).3. **L'effet août** — croisé avec `T2_N1` (même fenêtre un an plus tôt) pour   séparer la saisonnalité académique de notre propre travail.

In [ ]:
if not gsc_daily.empty:    serie = gsc_daily[gsc_daily["periode"] != "GLOBAL"].copy()    serie["date"] = pd.to_datetime(serie["date"])    quotidien = serie.groupby("date")["clicks"].sum()    moyenne, ecart = quotidien.mean(), quotidien.std()    anomalies = quotidien[(quotidien - moyenne).abs() > 2.5 * ecart]    fig, ax = plt.subplots(figsize=(11, 3.6))    ax.plot(quotidien.index, quotidien.values, color=viz.color_for("clics"),            linewidth=1.5, label="clics/jour")    ax.axhline(moyenne, color=viz.INK_MUTED, linestyle="--", linewidth=1,               label=f"moyenne {moyenne:,.0f}")    if len(anomalies):        ax.scatter(anomalies.index, anomalies.values, color=viz.DIVERGING["neg"],                   zorder=5, s=45, label=f"{len(anomalies)} anomalie(s)")    ax.set_title("Clics quotidiens et anomalies (seuil ±2,5 σ)")    ax.legend(loc="upper left")    fig.tight_layout()    viz.save(fig, "07-anomalies")    display(anomalies.to_frame("clics"))

In [ ]:
# Bot Direct GA4 : part du trafic Direct par période.if not ga4_channels.empty and "sessionDefaultChannelGroup" in ga4_channels.columns:    direct = ga4_channels.copy()    direct["part_pct"] = (direct["sessions"]                          / direct.groupby("periode")["sessions"].transform("sum") * 100)    tableau = direct.pivot_table(index="sessionDefaultChannelGroup",                                 columns="periode", values="part_pct").round(1)    display(tableau)    if "Direct" in tableau.index:        print("\nPart du Direct par période :")        print(tableau.loc["Direct"])        print("→ Une part qui remonte nettement en T2 = le bruit bot est revenu : "              "neutraliser le Direct avant toute comparaison d'audience.")else:    print("ga4_channels manquant — lancer extract_ga4.py")

In [ ]:
# Saisonnalité : T2 comparé à la même fenêtre un an plus tôt.if not gsc_totals.empty and {"T2", "T2_N1"} <= set(gsc_totals["periode"]):    saison = gsc_totals.set_index("periode").loc[["T2_N1", "T2"],                                                 ["clicks_30j", "impressions_30j", "ctr_pct", "position"]]    display(saison.round(1))    croissance = (saison.loc["T2", "clicks_30j"] / saison.loc["T2_N1", "clicks_30j"] - 1) * 100    print(f"\nCroissance à saison comparable (T2 vs T2 N-1) : {croissance:+.1f}%")    print("C'est ce chiffre-là qu'il faut citer au client, pas T1 → T2 brut : "          "T2 contient août, T1 non.")else:    print("T2_N1 absent — relancer extract_search_console.py --periods T2_N1")

## 9. Leads et ROI⚠️ Le rapport #1 montre que les conversions Vilber passent à 97 % par`contact_submitted` (41 sur 42) et **pas** par `/get-a-demo` (1 sur 42). Un ROIcalculé sur les seules demandes de démo sous-estimerait l'acquisition d'unfacteur ~40. On compte donc les deux, et on affiche la part organique.La valeur d'un lead est une **hypothèse à faire valider par Anaïs** avantpublication — elle n'est documentée nulle part dans Notion.

In [ ]:
VALEUR_LEAD_EUR = None   # ← à renseigner après validation par AnaïsCOUT_MENSUEL_EUR = 1100  # le brief dit 1 100 €/mois ; la fiche client Notion dit 1 200 €/mois — à trancherif not ga4_conv.empty:    conv = ga4_conv.copy()    conv["eventCount"] = pd.to_numeric(conv["eventCount"], errors="coerce")    par_periode = conv.pivot_table(index="eventName", columns="periode",                                   values="eventCount", aggfunc="sum")    display(par_periode)    if "firstUserMedium" in conv.columns:        organique = conv[conv["firstUserMedium"].astype(str).str.contains("organic", na=False)]        part = (organique.groupby("periode")["eventCount"].sum()                / conv.groupby("periode")["eventCount"].sum() * 100).round(1)        print("\nPart des conversions d'origine organique (%) :")        print(part)if not webflow.empty:    display(webflow)if VALEUR_LEAD_EUR is None:    print("\nROI non calculé : VALEUR_LEAD_EUR n'est pas renseignée. "          "Publier un ROI sur une valeur de lead inventée serait pire que de ne pas en publier.")

## 10. Export des tableaux du rapportÉcrit dans `report/figures/` (graphiques) et `data/consolidated/` (tables).Les tableaux ci-dessous se collent tels quels dans`report/rapport-vilber-bilan-3m.md`.

In [ ]:
def en_markdown(df, titre):    if df is None or df.empty:        return f"### {titre}\n\n_(données non disponibles)_\n"    return f"### {titre}\n\n" + df.to_markdown(index=False) + "\n"blocs = [    en_markdown(kpis, "Synthèse KPIs T0 / T1 / T2"),    en_markdown(deltas, "Deltas"),    en_markdown(branded, "Part des clics de marque"),    en_markdown(position_mix, "Distribution des impressions par tranche de position"),]sortie = FIGURES_DIR.parent / "tableaux-generes.md"sortie.write_text("\n\n".join(blocs), encoding="utf-8")print(f"tableaux écrits dans {sortie}")print("\nFigures disponibles :")for f in sorted(FIGURES_DIR.glob("*.png")):    print(" -", f.name)